In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
%pip install -q -U torch transformers accelerate huggingface_hub safetensors bitsandbytes
# https://huggingface.co/HuggingFaceTB/SmolLM2-360M
# https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct
# https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct

from transformers import AutoTokenizer
import torch

MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
# MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
# MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("Device :", DEVICE)
print("Dtype  :", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

print("Tokenizer chargé.")


In [ ]:
# https://huggingface.co/docs/transformers/quantization/bitsandbytes?bnb=4-bit
import torch
import copy
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)

model_quantized = copy.deepcopy(model_base)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

model_quantized.eval()

print("Modèle quantifié en 4 bits chargé.")


In [ ]:
# Mesurer perplexité et accuracy du prochain token
import math
import torch.nn.functional as F
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)

texts = [t for t in dataset["text"] if t.strip()][:500]
encoded = tokenizer("\n\n".join(texts), return_tensors="pt")["input_ids"][0]

context_length = getattr(model_base.config, "max_position_embeddings", 1024)
seq_len = min(context_length, 512)


# calculer la perplexité en pytorch
# https://huggingface.co/docs/transformers/en/perplexity
def quality_metrics(model, input_ids, seq_len):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    correct = 0

    with torch.inference_mode():
        for start in range(0, input_ids.numel() - 1, seq_len):
            chunk = input_ids[start : start + seq_len + 1]
            if chunk.numel() < 2:
                continue

            x = chunk[:-1].unsqueeze(0).to(DEVICE)
            y = chunk[1:].unsqueeze(0).to(DEVICE)

            logits = model(input_ids=x).logits
            logits = logits[:, :y.shape[1], :]

            total_nll += F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
                reduction="sum",
            ).item()

            predictions = logits.argmax(dim=-1)
            correct += (predictions == y).sum().item()
            total_tokens += y.numel()

    mean_nll = total_nll / total_tokens
    return {
        "perplexity": math.exp(mean_nll),
        "next_token_accuracy": correct / total_tokens,
    }

quality_base = quality_metrics(model_base, encoded, seq_len)
quality_quantized = quality_metrics(model_quantized, encoded, seq_len)

print("Base :", quality_base)
print("Quantifié:", quality_quantized)



In [ ]:
# Mesurer latence, tokens et mémoire
import numpy as np
import time

prompt = "Être ou ne pas être, telle est la question ?"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

MAX_NEW_TOKENS = 64
WARMUP = 3
REPEATS = 30

def benchmark_generation(model, inputs, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()

    def synchronize():
        if torch.cuda.is_available() and str(DEVICE).startswith("cuda"):
            torch.cuda.synchronize()

    with torch.inference_mode():
        for _ in range(WARMUP):
            _ = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )
    synchronize()

    latencies = []
    generated_counts = []

    with torch.inference_mode():
        for _ in range(REPEATS):
            synchronize()
            t0 = time.perf_counter()

            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )

            synchronize()
            latencies.append(time.perf_counter() - t0)
            generated_counts.append(output.shape[1] - inputs["input_ids"].shape[1])

    latencies = np.array(latencies)
    generated_counts = np.array(generated_counts)

    result = {
        "latency_p50_s": np.percentile(latencies, 50),
        "latency_p95_s": np.percentile(latencies, 95),
        "tokens_per_s": generated_counts.sum() / latencies.sum(),
    }

    return result

perf_base = benchmark_generation(model_base, inputs)
perf_pruned = benchmark_generation(model_quantized, inputs)

print("Base :", perf_base)
print("Quantifié:", perf_pruned)



In [ ]:
import os
import psutil

process = psutil.Process(os.getpid())

# https://psutil.io/#psutil.Process.memory_info
# https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory_allocated.html

def measure_resources():
    process_rss_mb = process.memory_info().rss / 1024**2
    virtual_memory = psutil.virtual_memory()
    swap_memory = psutil.swap_memory()
    cpu_percent = psutil.cpu_percent(interval=1)
    cpu_times = psutil.cpu_times()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        gpu_allocated_mb = torch.cuda.memory_allocated() / 1024**2
        gpu_reserved_mb = torch.cuda.memory_reserved() / 1024**2
        gpu_peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    else:
        gpu_allocated_mb = float("nan")
        gpu_reserved_mb = float("nan")
        gpu_peak_mb = float("nan")

    return {
        "ram_process_mb": process_rss_mb,
        "ram_system_total_mb": virtual_memory.total / 1024**2,
        "ram_system_available_mb": virtual_memory.available / 1024**2,
        "ram_system_percent": virtual_memory.percent,
        "swap_total_mb": swap_memory.total / 1024**2,
        "swap_used_mb": swap_memory.used / 1024**2,
        "swap_percent": swap_memory.percent,
        "cpu_percent": cpu_percent,
        "cpu_times_user_s": cpu_times.user,
        "cpu_times_system_s": cpu_times.system,
        "gpu_allocated_mb": gpu_allocated_mb,
        "gpu_reserved_mb": gpu_reserved_mb,
        "gpu_peak_mb": gpu_peak_mb,
    }


stats = measure_resources()
print(stats)




In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_quantized_model(bits):
    if bits == 4:
        config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=config,
            device_map="auto",
        )

    elif bits == 8:
        config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=config,
            device_map="auto",
        )

    elif bits in (16, "fp16", "bf16", "fp32"):
        dtype = {
            16: torch.float16,
            "fp16": torch.float16,
            "bf16": torch.bfloat16,
            "fp32": torch.float32,
        }[bits]

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=dtype,
            device_map="auto",
        )

    else:
        raise ValueError("bits doit être 4, 8, 16, 'bf16' ou 'fp32'")

    model.eval()
    return model


In [ ]:
# Comparer plusieurs taux d’élagage et tracer coût/performance
import pandas as pd

modes = [16, 8, 4, "bf16", "fp32"]

labels = {
    16: "fp16",
    8: "8-bit",
    4: "4-bit NF4",
    "bf16": "bfloat16",
    "fp32": "float32",
}

rows = []

for mode in modes:
    candidate = load_quantized_model(mode)
    device = candidate.get_input_embeddings().weight.device

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    q = quality_metrics(candidate, encoded, seq_len)
    p = benchmark_generation(candidate, inputs)
    m = measure_resources()

    rows.append({
        "quantification": labels[mode],
        **q,
        **p,
        **m,
    })

    del candidate

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("quantification_resultats")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "comparaison.csv"
results.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Graphique 1
fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(results["tokens_per_s"], results["perplexity"], marker="o")

for _, row in results.iterrows():
    ax1.annotate(
        str(row["quantification"]),
        (row["tokens_per_s"], row["perplexity"]),
        xytext=(5, 5),
        textcoords="offset points",
    )

ax1.set_xlabel("Débit (tokens/s, plus élevé = mieux)")
ax1.set_ylabel("Perplexité (plus bas = mieux)")
ax1.set_title("Compromis qualité / performance")
ax1.grid(True)

quality_path = output_dir / "compromis_qualite_performance.png"
fig1.savefig(quality_path, dpi=300, bbox_inches="tight")

# Graphique 2
fig2, ax2 = plt.subplots(figsize=(8, 5))

x = results["quantification"]

if "ram_process_mb" in results.columns:
    ax2.plot(x, results["ram_process_mb"], marker="o", label="RAM processus")

if "gpu_peak_mb" in results.columns and results["gpu_peak_mb"].notna().any():
    ax2.plot(x, results["gpu_peak_mb"], marker="s", label="Pic mémoire GPU")

ax2.set_xlabel("Élagage (%)")
ax2.set_ylabel("Mémoire (MB)")
ax2.set_title("Empreinte mémoire selon le taux d’élagage")
ax2.grid(True)
ax2.legend()

memory_path = output_dir / "empreinte_memoire.png"
fig2.savefig(memory_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"CSV enregistré : {csv_path.resolve()}")
print(f"Graphique qualité/performance : {quality_path.resolve()}")
print(f"Graphique mémoire : {memory_path.resolve()}")

